# Momentum-Based Sector Rotation Strategy

**Strategy rules:**
- Universe: 11 GICS sector ETFs
- Signal: trailing 6-month cumulative return (momentum)
- Portfolio: equal-weight top-3 sectors by signal, monthly rebalance
- Benchmark: SPY buy-and-hold

**Metrics:** Total return, annualized return, annualized vol, Sharpe ratio, max drawdown, Calmar ratio.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

SECTOR_ETFS = ["XLK", "XLF", "XLV", "XLY", "XLP", "XLE", "XLI", "XLB", "XLU", "XLRE", "XLC"]
START_DATE = "2010-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
LOOKBACK_MONTHS = 6
TOP_N = 3
print(f"Strategy: Top-{TOP_N} sectors by {LOOKBACK_MONTHS}m momentum, monthly rebalance")

In [ ]:
all_tickers = SECTOR_ETFS + ["SPY"]
prices = yf.download(all_tickers, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"].dropna(how="all")
monthly = prices.resample("ME").last()
print(f"Monthly prices: {monthly.shape[0]} months x {monthly.shape[1]} tickers")

## Strategy Implementation

In [ ]:
sector_px = monthly[SECTOR_ETFS].dropna(how="all")
sector_ret = sector_px.pct_change()
momentum = sector_px.pct_change(LOOKBACK_MONTHS)

strat_returns, strat_dates, selected_history = [], [], []

for i in range(LOOKBACK_MONTHS + 1, len(sector_px)):
    scores = momentum.iloc[i - 1].dropna()
    available = [t for t in scores.index if t in sector_ret.columns]
    if len(available) < TOP_N:
        continue
    top3 = scores[available].nlargest(TOP_N).index.tolist()
    month_ret = sector_ret.iloc[i][top3].mean()
    strat_returns.append(month_ret)
    strat_dates.append(sector_px.index[i])
    selected_history.append(top3)

strat_ret = pd.Series(strat_returns, index=strat_dates, name="Strategy")
spy_ret = monthly["SPY"].pct_change().reindex(strat_ret.index)
print(f"Strategy months: {len(strat_ret)}")
print(f"Most recently selected: {selected_history[-1]}")

## Performance Metrics

In [ ]:
def performance_metrics(monthly_returns, name, rf_annual=0.04):
    cum = (1 + monthly_returns).cumprod()
    total = cum.iloc[-1] - 1
    n_months = len(monthly_returns)
    ann_ret = (1 + total) ** (12 / n_months) - 1
    ann_vol = monthly_returns.std() * np.sqrt(12)
    rf_m = (1 + rf_annual) ** (1/12) - 1
    sharpe = (monthly_returns.mean() - rf_m) / monthly_returns.std() * np.sqrt(12)
    dd = (cum - cum.cummax()) / cum.cummax()
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else float("nan")
    print(f"
{name}:")
    print(f"  Total Return:    {total:>8.1%}")
    print(f"  Ann. Return:     {ann_ret:>8.1%}")
    print(f"  Ann. Volatility: {ann_vol:>8.1%}")
    print(f"  Sharpe Ratio:    {sharpe:>8.2f}")
    print(f"  Max Drawdown:    {max_dd:>8.1%}")
    print(f"  Calmar Ratio:    {calmar:>8.2f}")
    return cum, dd

cum_strat, dd_strat = performance_metrics(strat_ret, "Momentum Rotation")
cum_spy, dd_spy = performance_metrics(spy_ret.dropna(), "SPY Buy & Hold")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(cum_strat, label=f"Momentum Rotation (Top {TOP_N})", color="#2196F3", lw=2)
axes[0].plot(cum_spy, label="SPY Buy & Hold", color="#FF5722", lw=1.5, ls="--")
axes[0].set_title("Equity Curves: Momentum Sector Rotation vs SPY")
axes[0].set_ylabel("Cumulative Return")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].fill_between(dd_strat.index, dd_strat.values, 0, color="#2196F3", alpha=0.4, label="Strategy")
axes[1].fill_between(dd_spy.index, dd_spy.values, 0, color="#FF5722", alpha=0.4, label="SPY")
axes[1].set_title("Drawdown Comparison")
axes[1].set_ylabel("Drawdown")
axes[1].legend()
axes[1].grid(alpha=0.3)

from collections import Counter
freq = Counter(s for month in selected_history for s in month)
sectors_s = sorted(freq, key=lambda s: freq[s], reverse=True)
axes[2].bar(sectors_s, [freq[s] for s in sectors_s], color="steelblue", alpha=0.8)
axes[2].set_title("Sector Selection Frequency")
axes[2].set_ylabel("Months Selected")
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusions

- Momentum rotation systematically overweights recent outperformers.
- Monthly rebalancing with 6-month lookback captures intermediate-term momentum.
- The strategy may underperform during sharp momentum reversals (lag of one month).
- No transaction costs modeled — sector ETF costs are minimal but exist.